# DroneSplat + LLMs (Difix3D+ / ArtiFixer) — BOSCH Setup
Run once with the **JupyterHub default kernel** (Python 3.11).  
Creates the `dronesplat_llms` venv, compiles 3DGS CUDA extensions, installs generative 3D packages (`diffusers`, `transformers`, `open3d`), and downloads model weights.

**What this installs:**
1. DroneSplat baseline (3DGS + SAM2 distractor masking + CUDA rasterizers)
2. NVIDIA Difix3D+ (Single-step diffusion 3D artifact restoration, reference-conditioned)
3. NVIDIA ArtiFixer (Wan2.1 14B Auto-Regressive Video Diffusion for unobserved BST angles)

| Cell | Purpose |
|------|---------|
| `c00_proxy` | Set BOSCH proxy env vars |
| `c01_config` | Paths and constants |
| `c02_create_env` | Create `dronesplat_llms` venv (python -m venv) |
| `c03_torch` | Install PyTorch 2.6.0 + CUDA 12.6 |
| `c04_pip_req` | Install base requirement packages (`open3d`, `matplotlib`, etc.) & force-reinstall torch cu126 |
| `c05_cuda_ext` | Search `nvcc`, clone submodules + GLM, compile wheel & install (`simple-knn`, `diff-gaussian-rasterization`) |
| `c06_sam2` | Install SAM2 video tracker |
| `c07_generative_req` | Install `diffusers`, `transformers`, `accelerate`, `scipy`, `lpips` |
| `c08_weights` | Download `nvidia/difix_ref` weights (reference-conditioned) |
| `c09_kernel` | Register `dronesplat_llms` as a Jupyter kernel |
| `c10_verify` | Import verification check — all rows must PASS |

## c00 — Proxy
Sets the BOSCH proxy for all subsequent pip / git / HuggingFace downloads. Run this **first**.

In [ ]:
# ── Proxy (required for pip / HuggingFace downloads) ─────────────────────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

print(f'Proxy set: {PROXY}')

## c01 — Paths & Config
Defines path variables and verifies required repository directories.

In [ ]:
# ── Paths & constants ─────────────────────────────────────────────────────────
import os
HOME           = os.path.expanduser('~')
RACE_DIR       = f'{HOME}/Race-AI-2026'
DRONESPLAT_DIR = f'{RACE_DIR}/projects/DroneSplat'
DIFIX_DIR      = f'{RACE_DIR}/projects/Difix3D'
ARTIFIXER_DIR  = f'{RACE_DIR}/projects/ArtiFixer'
ENV_NAME       = 'dronesplat_llms'

print(f'HOME          : {HOME}')
print(f'DRONESPLAT_DIR: {DRONESPLAT_DIR}')
print(f'DIFIX_DIR     : {DIFIX_DIR}')
print(f'ARTIFIXER_DIR : {ARTIFIXER_DIR}')

assert os.path.isdir(DRONESPLAT_DIR), f'DroneSplat not found at {DRONESPLAT_DIR}'
assert os.path.isdir(DIFIX_DIR), f'Difix3D not found at {DIFIX_DIR}'
assert os.path.isdir(ARTIFIXER_DIR), f'ArtiFixer not found at {ARTIFIXER_DIR}'

## c02 — Create venv
Creates `dronesplat_llms` using `python -m venv`.

In [ ]:
# ── Create dronesplat_llms venv ───────────────────────────────────────────────
import subprocess, json, os, sys, shutil

HOME     = os.path.expanduser('~')
PROXY    = 'http://rb-proxy-sl.bosch.com:8080'
ENV_NAME = 'dronesplat_llms'

os.environ.update({'http_proxy': PROXY, 'https_proxy': PROXY,
                   'HTTP_PROXY': PROXY, 'HTTPS_PROXY': PROXY})

r_info   = subprocess.run(['conda', 'info', '--json'], capture_output=True, text=True)
info     = json.loads(r_info.stdout)
envs_dir = info.get('envs_dirs', [f'{HOME}/.conda/envs'])[0]
dronesplat_env    = os.path.join(envs_dir, ENV_NAME)
dronesplat_python = os.path.join(dronesplat_env, 'bin', 'python')
dronesplat_pip    = os.path.join(dronesplat_env, 'bin', 'pip')

if os.path.isfile(dronesplat_python):
    print(f'Env already exists at {dronesplat_env}')
else:
    candidates = [
        shutil.which('python3.11'), '/opt/conda/bin/python3.11',
        shutil.which('python3.10'), '/opt/conda/bin/python3.10', sys.executable
    ]
    base_python = next((p for p in candidates if p and os.path.isfile(p)), sys.executable)
    print(f'Base Python : {base_python}')
    os.makedirs(envs_dir, exist_ok=True)
    r_venv = subprocess.run([base_python, '-m', 'venv', dronesplat_env], capture_output=True, text=True)
    if r_venv.returncode != 0:
        raise RuntimeError(f'venv creation failed: {r_venv.stderr}')
    subprocess.run([dronesplat_pip, 'install', '--upgrade', 'pip', '--proxy', PROXY], capture_output=True, text=True)
    print(f'Created venv: {dronesplat_env}')

print(f'\ndronesplat_llms env   : {dronesplat_env}')
print(f'dronesplat_llms python: {dronesplat_python}')

## c03 — Install PyTorch
Installs PyTorch 2.6.0 with CUDA 12.6 support.

In [ ]:
# ── Install PyTorch ───────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(
    [dronesplat_pip, 'install', 'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
     '--index-url', 'https://download.pytorch.org/whl/cu126', '--proxy', PROXY],
    capture_output=True, text=True
)
print('PyTorch Install Output:', 'OK' if r.returncode == 0 else r.stderr[-500:])

## c04 — Install base python requirements
Installs dependencies including `open3d`, `matplotlib`, `scikit-image` & force-reinstalls torch cu126 to prevent version mismatch.

In [ ]:
# ── Base requirements & wheel/setuptools ─────────────────────────────────────
import subprocess
base_deps = ['wheel', 'setuptools', 'ipykernel', 'tensorboard', 'pandas', 'plyfile', 'tqdm', 'ninja', 'scikit-image', 'open3d', 'matplotlib']
r = subprocess.run([dronesplat_pip, 'install'] + base_deps + ['--proxy', PROXY], capture_output=True, text=True)
print('Base deps output:', 'OK' if r.returncode == 0 else r.stderr[-300:])

# Force-reinstall torch cu126
r3 = subprocess.run([
    dronesplat_pip, 'install', '--force-reinstall',
    'torch==2.6.0', 'torchvision==0.21.0',
    '--index-url', 'https://download.pytorch.org/whl/cu126', '--proxy', PROXY
], capture_output=True, text=True)
print('Force reinstall torch:', 'OK' if r3.returncode == 0 else r3.stderr[-300:])

## c05 — Build CUDA Extensions (simple-knn + diff-gaussian-rasterization)
Locates system `nvcc`, clones submodule mirrors (`camenduru/simple-knn` & `graphdeco-inria/diff-gaussian-rasterization` + GLM), compiles wheels with `setup.py bdist_wheel`, and installs them.

In [ ]:
# ── Build CUDA extensions: simple-knn + diff-gaussian-rasterization ───────────
import subprocess, os, sys, glob

# ── Search nvcc ───────────────────────────────────────────────────────────────
search_script = r'''
which nvcc 2>/dev/null && exit 0
for init in /etc/profile /etc/profile.d/modules.sh \
            /usr/share/lmod/lmod/init/bash /usr/share/lmod/lmod/init/sh; do
    [ -f "$init" ] && source "$init" 2>/dev/null
done
for mod in cuda/12.6 cuda/12 cuda CUDA/12.6 CUDA cuda-12.6 cuda-12; do
    module load "$mod" 2>/dev/null
    nv=$(which nvcc 2>/dev/null); [ -n "$nv" ] && echo "$nv" && exit 0
done
for base in /fs /work /gpfs /scratch /software /apps /appl /tools /opt/software /usr/local; do
    [ -d "$base" ] || continue
    result=$(find "$base" -name nvcc -type f -maxdepth 8 2>/dev/null | head -1)
    [ -n "$result" ] && echo "$result" && exit 0
done
'''
r_search = subprocess.run(['bash', '-c', search_script], capture_output=True, text=True, timeout=90)
nvcc_path = r_search.stdout.strip()
print(f'nvcc found: {repr(nvcc_path)}')
if not nvcc_path or not os.path.isfile(nvcc_path):
    raise RuntimeError('nvcc not found — check CUDA module availability')

cuda_home = os.path.dirname(os.path.dirname(nvcc_path))
os.environ['CUDA_HOME'] = cuda_home

r_find_rt = subprocess.run(['find', dronesplat_env, '-name', 'libcudart*', '-type', 'f'],
                           capture_output=True, text=True, timeout=30)
rt_libs = [p.strip() for p in r_find_rt.stdout.strip().split('\n') if p.strip()]
cuda_rt_lib = os.path.dirname(rt_libs[0]) if rt_libs else ''
ld_path = f'{cuda_home}/lib64:{cuda_rt_lib}' if cuda_rt_lib else f'{cuda_home}/lib64'
sys_cc  = subprocess.run(['which', 'gcc'], capture_output=True, text=True).stdout.strip() or '/usr/bin/gcc'
sys_cxx = subprocess.run(['which', 'g++'], capture_output=True, text=True).stdout.strip() or '/usr/bin/g++'
cc = '8.0'
print(f'CUDA_HOME : {cuda_home}')
print(f'CC/CXX    : {sys_cc} / {sys_cxx}')

SUBMODULES_DIR = f'{DRONESPLAT_DIR}/submodules'
os.makedirs(SUBMODULES_DIR, exist_ok=True)
clone_env = {**os.environ}
git_proxy = ['-c', f'http.proxy={PROXY}', '-c', f'https.proxy={PROXY}']

# Clone repositories
repos = [
    ('simple-knn',                  'https://github.com/camenduru/simple-knn'),
    ('diff-gaussian-rasterization', 'https://github.com/graphdeco-inria/diff-gaussian-rasterization.git'),
    ('sam2',                        'https://github.com/facebookresearch/sam2.git'),
]
for name, url in repos:
    tgt = os.path.join(SUBMODULES_DIR, name)
    if os.path.isdir(os.path.join(tgt, '.git')) or (os.path.isdir(tgt) and os.listdir(tgt)):
        print(f'{name}: already present — skip')
    else:
        print(f'Cloning {name} ...')
        r = subprocess.run(['git'] + git_proxy + ['clone', url, tgt, '--depth=1'],
                           capture_output=True, text=True, env=clone_env, timeout=300)
        print(f'{name}: cloned OK' if r.returncode == 0 else f'Clone failed: {r.stderr[-300:]}')

# GLM headers for diff-gaussian-rasterization
glm_dir = os.path.join(SUBMODULES_DIR, 'diff-gaussian-rasterization', 'third_party', 'glm')
if not (os.path.isdir(glm_dir) and os.listdir(glm_dir)):
    os.makedirs(os.path.dirname(glm_dir), exist_ok=True)
    r = subprocess.run(['git'] + git_proxy + ['clone', 'https://github.com/g-truc/glm.git', glm_dir, '--depth=1'],
                       capture_output=True, text=True, env=clone_env, timeout=300)
    print('GLM cloned OK' if r.returncode == 0 else r.stderr[-300:])

# Build CUDA extension wheels
def build_ext(name, src_dir):
    print(f'\n=== Building {name} ===')
    if not os.path.isdir(src_dir):
        print(f'ERROR: source dir not found: {src_dir}'); return False
    build_cmd = (
        f'cd {src_dir} && rm -rf build dist && '
        f'CUDA_HOME={cuda_home} PATH={cuda_home}/bin:$PATH '
        f'LD_LIBRARY_PATH={ld_path}:$LD_LIBRARY_PATH '
        f'TORCH_CUDA_ARCH_LIST={cc} CC={sys_cc} CXX={sys_cxx} '
        f'{dronesplat_python} setup.py bdist_wheel 2>&1'
    )
    r = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('BUILD STDERR:', r.stderr[-1000:]); return False
    wheels = glob.glob(f'{src_dir}/dist/*.whl')
    if not wheels:
        print(f'ERROR: No wheel in {src_dir}/dist/'); return False
    ri = subprocess.run([dronesplat_pip, 'install', wheels[0], '--no-deps', '--force-reinstall'],
                        capture_output=True, text=True)
    if ri.returncode != 0:
        print('INSTALL STDERR:', ri.stderr[-500:]); return False
    print(f'{name}: OK'); return True

exts = [
    ('simple-knn',                  f'{SUBMODULES_DIR}/simple-knn'),
    ('diff-gaussian-rasterization', f'{SUBMODULES_DIR}/diff-gaussian-rasterization'),
]
results = {name: build_ext(name, src) for name, src in exts}
print('\n=== Build Summary ===')
for name, ok in results.items():
    print(f'  {name:35s}: {"OK" if ok else "FAILED"}')

## c06 — Install SAM2
Installs Segment Anything 2 (SAM2) via `pip install -e`.

In [ ]:
# ── Install SAM2 ──────────────────────────────────────────────────────────────
SAM2_DIR = f'{DRONESPLAT_DIR}/submodules/sam2'
if os.path.isdir(SAM2_DIR):
    r = subprocess.run([dronesplat_pip, 'install', '-e', SAM2_DIR, '--no-deps', '--proxy', PROXY], capture_output=True, text=True)
    print('SAM2 install:', 'OK' if r.returncode == 0 else r.stderr[-300:])

## c07 — Install Generative 3D Packages (Difix3D+ & ArtiFixer)
Installs `diffusers`, `transformers`, `accelerate`, `scipy`, and `lpips`.

In [ ]:
# ── Install Generative 3D Requirements ───────────────────────────────────────
gen_pkgs = ['diffusers', 'transformers', 'accelerate', 'scipy', 'lpips', 'huggingface_hub', 'pillow', 'open3d']
r = subprocess.run([dronesplat_pip, 'install'] + gen_pkgs + ['--proxy', PROXY], capture_output=True, text=True)
print('Generative dependencies install:', 'OK' if r.returncode == 0 else r.stderr[-400:])

## c08 — Model Checkpoint Downloads
Downloads `nvidia/difix_ref` weights (reference-conditioned Difix3D+, the in-loop distillation trainer's default checkpoint) to local cache.

In [ ]:
# ── Model Checkpoint Downloads ────────────────────────────────────────────────
print('Downloading nvidia/difix_ref pipeline weights (reference-conditioned, used by default)...')
dl_cmd = [
    dronesplat_python, '-c',
    "from huggingface_hub import snapshot_download; snapshot_download('nvidia/difix_ref')"
]
r = subprocess.run(dl_cmd, capture_output=True, text=True)
print('Difix3D+ (ref) Download:', 'OK' if r.returncode == 0 else r.stderr[-300:])

## c09 — Register Jupyter Kernel
Registers `dronesplat_llms` as an available Jupyter kernel.

In [ ]:
# ── Register Jupyter Kernel ───────────────────────────────────────────────────
subprocess.run([dronesplat_pip, 'install', 'ipykernel', '--proxy', PROXY], capture_output=True, text=True)
r_k = subprocess.run([
    dronesplat_python, '-m', 'ipykernel', 'install',
    '--user', '--name', 'dronesplat_llms',
    '--display-name', 'Python (dronesplat_llms)'
], capture_output=True, text=True)
print('Kernel Register Output:', 'OK' if r_k.returncode == 0 else r_k.stderr[-300:])

## c10 — Verify Imports
Validates all core packages in `dronesplat_llms`.

In [ ]:
# ── Verify Imports ───────────────────────────────────────────────────────────
verify_script = """
import torch, diff_gaussian_rasterization, simple_knn, sam2, diffusers, transformers, open3d
print('PyTorch CUDA:', torch.cuda.is_available())
print('Rasterizer  : OK')
print('simple-knn  : OK')
print('SAM2        : OK')
print('Diffusers   : OK')
print('Open3D      : OK')
"""
r_v = subprocess.run([dronesplat_python, '-c', verify_script], capture_output=True, text=True)
print(r_v.stdout if r_v.returncode == 0 else f'Verification Failed:\n{r_v.stderr}')